In [3]:
from OSMPythonTools.overpass import Overpass, overpassQueryBuilder
import pandas as pd

In [4]:
overpass = Overpass()

In [40]:
def get_volcanoes(lat_band=30):
    query = overpassQueryBuilder(
        bbox=[-lat_band, -180, lat_band, 180],   # [south, west, north, east]
        elementType=['node', 'way', 'relation'],
        selector=['"natural"="volcano"'],
        includeCenter=True,   # gives ways/relations a representative point
        out='body',
    )
    result = overpass.query(query, timeout=180)

    rows = []
    for el in result.elements():
        if el.type() == 'node':
            lat, lon = el.lat(), el.lon()
        else:
            lat, lon = el.centerLat(), el.centerLon()
        rows.append({
            'osm_type': el.type(),
            'osm_id': el.id(),
            'name': el.tag('name'),
            'name_en': el.tag('name:en'),
            'lat': lat,
            'lon': lon,
            'elevation': el.tag('ele'),
            'volcano_type': el.tag('volcano:type'),
            'volcano_status': el.tag('volcano:status'),
            'wikidata': el.tag('wikidata'),
        })
    return pd.DataFrame(rows)

In [54]:
df

,osm_type,osm_id,name,name_en,lat,lon,elevation,volcano_type,volcano_status,wikidata
0,node,29017255,Volcán de San Antonio,NaN,28.482773,-17.848392,656,NaN,dormant,Q955177
1,node,58959434,Chimborazo,NaN,-1.469082,-78.815950,6268,stratovolcano,dormant,Q14081
2,node,58981085,Illiniza Sur,NaN,-0.663012,-78.715850,5248,stratovolcano,extinct,Q734280
3,node,58989127,Popocatépetl,NaN,19.023405,-98.623437,5452,stratovolcano,active,Q181157
4,node,58991090,Iztaccíhuatl,NaN,19.170205,-98.637591,5230,stratovolcano,dormant,Q664059
...,...,...,...,...,...,...,...,...,...,...
10761,way,1545489827,Kawah Keramikan,Keramikan Crater,NaN,NaN,NaN,maar,active,NaN
10762,relation,5740787,Tukuran Volcanic Field,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10763,relation,9841413,Mount Bagana,NaN,NaN,NaN,NaN,NaN,NaN,Q799827
10764,relation,15002257,Bilate River Field,NaN,NaN,NaN,1700,NaN,extinct,Q860406


In [53]:
df[df['volcano_status'].isna()]

,osm_type,osm_id,name,name_en,lat,lon,elevation,volcano_type,volcano_status,wikidata
6,node,96183752,Nevado Ojos del Salado,Ojos del Salado,-27.109537,-68.541483,6891,stratovolcano,NaN,Q233836
8,node,96184596,NaN,NaN,-25.346645,-68.522485,NaN,NaN,NaN,NaN
15,node,96304728,Cerro Araral,NaN,-21.604440,-68.180480,5688,NaN,NaN,Q5064848
17,node,96305028,Cerro Ascotán,NaN,-21.678299,-68.117825,5470,NaN,NaN,Q21044357
18,node,96305270,Cerro Inacaliri,NaN,-21.980605,-68.071500,5628,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
10758,way,1259108622,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10759,way,1278065571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10762,relation,5740787,Tukuran Volcanic Field,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10763,relation,9841413,Mount Bagana,NaN,NaN,NaN,NaN,NaN,NaN,Q799827


### Ok, it looks like there's a lot of volcanoes where the volcano status tag is missing...
Maybe we need a separate DB to filter... Whatever, for now, let's roll with this

In [57]:
df[df['volcano_status'] == 'active'].sample(5)

,osm_type,osm_id,name,name_en,lat,lon,elevation,volcano_type,volcano_status,wikidata
5473,node,276219356,Volcan Turrialba,NaN,10.019322,-83.763153,3329,stratovolcano,active,Q546876
2043,node,3262499858,Mount Garet,NaN,-14.284594,167.517809,797,stratovolcano,active,Q1189723
1346,node,535314679,Volcán Putana,NaN,-22.557295,-67.853086,5890,stratovolcano,active,Q3411049
6793,node,548239934,Gunung Sirung,Mount Sirung,-8.499553,124.127978,862,NaN,active,Q1322666
4320,node,12039745082,Teahitia,NaN,-17.569991,-148.850055,-1600,NaN,active,NaN
